# Day 5 — Query Store & DMV Diagnostics

**Date:** 2026-09-23
**Database:** ProcurementDB
**Objective:** Identify top CPU and IO-consuming queries using DMVs and Query Store.

## 1. Top CPU-Consuming Queries

Querying `sys.dm_exec_query_stats` ordered by `total_worker_time` reveals the most CPU-intensive query patterns.

In [ ]:
-- ============================================
-- DAY 5 WORKLOADS — run these first to populate DMV stats
-- ============================================

-- Workload 1: Heavy aggregation
SELECT V.Region, V.VendorName,
       COUNT(PO.POID) AS Orders,
       SUM(PO.OrderAmount) AS Spend
FROM dbo.Vendors V
INNER JOIN dbo.PurchaseOrders PO ON V.VendorID = PO.VendorID
GROUP BY V.Region, V.VendorName;
GO 15

-- Workload 2: Status filter (uses Day 4 covering index)
SELECT POID, OrderAmount 
FROM dbo.PurchaseOrders 
WHERE Status = 'Pending';
GO 25

-- Workload 3: Date range scan
SELECT POID, OrderAmount, OrderDate
FROM dbo.PurchaseOrders
WHERE OrderDate >= '2026-01-01';
GO 5

-- Workload 4: Two-table JOIN with aggregation
SELECT PO.POID, SUM(LI.Quantity * LI.UnitCost) AS LineTotal
FROM dbo.PurchaseOrders PO
INNER JOIN dbo.LineItems LI ON PO.POID = LI.POID
WHERE PO.Status = 'Approved'
GROUP BY PO.POID;
GO 10

-- Workload 5: Correlated subquery (intentionally slow — our "guilty" query)
SELECT V.VendorID, V.VendorName,
    (SELECT SUM(OrderAmount) FROM dbo.PurchaseOrders 
     WHERE VendorID = V.VendorID) AS TotalSpend
FROM dbo.Vendors V;
GO 5

In [7]:
SELECT TOP 10
    qs.execution_count                                        AS Executions,
    qs.total_worker_time / 1000.0                             AS TotalCPU_ms,
    qs.total_worker_time / qs.execution_count / 1000.0        AS AvgCPU_ms,
    qs.total_logical_reads / qs.execution_count               AS AvgReads,
    qs.total_elapsed_time / qs.execution_count / 1000.0       AS AvgElapsed_ms,
    SUBSTRING(st.text, (qs.statement_start_offset/2)+1,
        ((CASE qs.statement_end_offset 
            WHEN -1 THEN DATALENGTH(st.text)
            ELSE qs.statement_end_offset END - qs.statement_start_offset)/2)+1
    ) AS QueryText
FROM sys.dm_exec_query_stats qs
CROSS APPLY sys.dm_exec_sql_text(qs.sql_handle) st
WHERE (st.text LIKE '%PurchaseOrders%' 
    OR st.text LIKE '%LineItems%'
    OR st.text LIKE '%Vendors%'
    OR st.text LIKE '%ApprovalLogs%')
  AND st.text NOT LIKE '%sys.%'
ORDER BY qs.total_worker_time DESC;

(5 rows affected)

Executions | TotalCPU_ms | AvgCPU_ms | AvgReads | AvgElapsed_ms | QueryText                                                                                                                                                                                               
-----------+-------------+-----------+----------+---------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
15         | 37.440000   | 2.496000  | 55       | 7.618000      | SELECT V.Region, V.VendorName,
       COUNT(PO.POID) AS Orders,
       SUM(PO.OrderAmount) AS Spend
FROM dbo.Vendors V
INNER JOIN dbo.PurchaseOrders PO ON V.VendorID = PO.VendorID
GROUP BY V.Region, V.VendorName
5          | 11.897000   | 2.379000  | 80       | 2.379000      | SELECT V.VendorID, V.VendorName,
    (SELECT SUM(OrderAmount) FROM dbo.PurchaseOrders 
     WHERE Vendo

### Finding — Top CPU Consumers

| Rank | Query | Execs | AvgCPU_ms | AvgReads |
| :--- | :--- | :--- | :--- | :--- |
| 1 | Aggregation (JOIN + GROUP BY) | 15 | **2.496** | 55 |
| 2 | Correlated subquery | 5 | 2.379 | 80 |
| 3 | Date range scan | 5 | 1.799 | 35 |

- **Top CPU:** Aggregation at 2.496 ms — CPU-bound due to sort + hash aggregation
- **Runner-up:** Correlated subquery at 2.379 ms — but with **80 avg reads** (IO-heavy)
- **Takeaway:** Nearly identical CPU costs, very different IO profiles

## 2. Top IO-Consuming Queries

Ordered by `total_logical_reads` — reveals IO-heavy queries that might be CPU-light but stress disk and memory.

In [8]:
SELECT TOP 5
    qs.execution_count                                   AS Executions,
    qs.total_logical_reads / qs.execution_count          AS AvgReads,
    qs.total_logical_reads                               AS TotalReads,
    qs.total_worker_time / qs.execution_count / 1000.0   AS AvgCPU_ms,
    SUBSTRING(st.text, (qs.statement_start_offset/2)+1,
        ((CASE qs.statement_end_offset 
            WHEN -1 THEN DATALENGTH(st.text)
            ELSE qs.statement_end_offset END - qs.statement_start_offset)/2)+1
    ) AS QueryText
FROM sys.dm_exec_query_stats qs
CROSS APPLY sys.dm_exec_sql_text(qs.sql_handle) st
WHERE (st.text LIKE '%PurchaseOrders%' 
    OR st.text LIKE '%LineItems%'
    OR st.text LIKE '%Vendors%'
    OR st.text LIKE '%ApprovalLogs%')
  AND st.text NOT LIKE '%sys.%'
ORDER BY qs.total_logical_reads DESC;

(5 rows affected)

Executions | AvgReads | TotalReads | AvgCPU_ms | QueryText                                                                                                                                                                                               
-----------+----------+------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
15         | 55       | 827        | 2.496000  | SELECT V.Region, V.VendorName,
       COUNT(PO.POID) AS Orders,
       SUM(PO.OrderAmount) AS Spend
FROM dbo.Vendors V
INNER JOIN dbo.PurchaseOrders PO ON V.VendorID = PO.VendorID
GROUP BY V.Region, V.VendorName
5          | 80       | 403        | 2.379000  | SELECT V.VendorID, V.VendorName,
    (SELECT SUM(OrderAmount) FROM dbo.PurchaseOrders 
     WHERE VendorID = V.VendorID) AS TotalSpend
FROM dbo.Vendors V                  

### Finding — Top IO Consumers (by TotalReads)

| Rank | Query | Execs | AvgReads | TotalReads |
| :--- | :--- | :--- | :--- | :--- |
| 1 | Aggregation | 15 | 55 | **827** |
| 2 | Correlated subquery | 5 | 80 | 403 |
| 3 | Status filter | 25 | 10 | 250 |
| 4 | Date range | 5 | 35 | 175 |
| 5 | Line items JOIN | 10 | 8 | 80 |

**Two ranking methods reveal different priorities:**

- **By TotalReads:** Aggregation leads (ran 15x → 827 total)
- **By AvgReads:** Correlated subquery leads (80 per execution)
- **Business interpretation:**
  - TotalReads → system throughput impact
  - AvgReads → individual request latency impact
- **Day 4 covering index proven:** Status filter = only 10 avg reads at runtime

## 3. Key Takeaways

| Insight | Evidence |
| :--- | :--- |
| CPU and IO are different metrics | Correlated subquery = 80 reads, only 2.38 ms CPU |
| Aggregation is CPU-bound | Full scan + GROUP BY = 2.50 ms CPU |
| Correlated subqueries are N+1 anti-patterns | ~10 scans per execution, one per vendor |
| Covering index works at runtime | Status filter = 10 reads (matches Day 4 measurement) |
| DMVs are ephemeral | Stats reset when DB pauses overnight |
| IntelliSense pollutes DMV stats | Filter by base table name to isolate real workload |

---

**Tools used:** VS Code + MSSQL extension (`.ipynb` renders on GitHub)

**Lessons learned:**

1. **Bracket-matching bug:** Cached SQL text may use `[dbo].[TableName]` — matching `LIKE '%dbo.TableName%'` misses it. Always match on the base name.
2. **DMVs are live; Query Store persists:** DMVs reset when the DB pauses; Query Store survives restarts and provides historical context.
3. **IntelliSense pollutes DMV output:** VS Code runs hundreds of metadata queries that dominate top-CPU lists. Filter by base table names.
4. **Total vs Average metrics:** Total reads show throughput impact; average reads show per-request latency. Check both to prioritize correctly.

---

**Status:** Complete — 2026-09-23